<a href="https://colab.research.google.com/github/ah1ahwon/seoul_mobility/blob/main/Seoul_Mobility_Full_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Seoul 2030 Mobility — 전체 파이프라인 (다운로드 → 분석 → 시각화)

이 노트북은 Google Colab에서 **처음부터 끝까지** 실행하는 파이프라인입니다.

| 단계 | 내용 |
|---|---|
| Step 1 | Google Drive 마운트 |
| Step 2 | GitHub 코드 클론 |
| Step 3 | 패키지 설치 |
| Step 4 | 데이터 직접 다운로드 → Drive 저장 |
| Step 5 | 분석 실행 |
| Step 6 | 결과 Drive 저장 |
| Step 7 | 시각화 |

**소요 시간 예상:** 다운로드 ~60분 + 분석 ~20분 + 시각화 ~5분

**Drive 필요 용량:** 약 4~5GB (월말 스냅샷 39개 + 3월 일별 30개 + 보조 파일)

## Step 1. Google Drive 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

# ▼ Drive 내 저장 폴더 (필요시 변경)
DRIVE_BASE   = Path("/content/drive/MyDrive/seoul_mobility")
DRIVE_RAW    = DRIVE_BASE / "raw"
DRIVE_OUTPUT = DRIVE_BASE / "output"

DRIVE_RAW.mkdir(parents=True, exist_ok=True)
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print("Drive 경로 준비 완료:", DRIVE_RAW)

Mounted at /content/drive
Drive 경로 준비 완료: /content/drive/MyDrive/seoul_mobility/raw


## Step 2. GitHub에서 코드 클론

In [3]:
REPO_URL = "https://github.com/ah1ahwon/seoul_mobility.git"
REPO_DIR = "/content/seoul_mobility"

if Path(REPO_DIR).exists():
    !git -C {REPO_DIR} pull --quiet
    print("최신 코드로 업데이트 완료")
else:
    !git clone --quiet {REPO_URL} {REPO_DIR}
    print("코드 클론 완료")

%cd {REPO_DIR}

코드 클론 완료
/content/seoul_mobility


## Step 3. 패키지 설치

In [4]:
!pip install -q -r requirements.txt
print("패키지 설치 완료")

패키지 설치 완료


## Step 4. 데이터 다운로드 → Google Drive 저장

서울 공공데이터에서 직접 Drive로 다운로드합니다. 이미 있는 파일은 건너뜁니다.

- 생활이동 월말 스냅샷 39개 (2023-01 ~ 2026-03)
- 2026년 3월 일별 30개
- 보조 데이터 4종 (지하철, 버스, 행정동, 시민생활)

In [ ]:
import subprocess
import time
import csv
import zipfile as _zipfile

REFERER_LIVING = "https://data.seoul.go.kr/dataList/OA-22299/F/1/datasetView.do"
DL_URL = "https://datafile.seoul.go.kr/bigfile/iot/inf/nio_download.do?useCache=false"

def _is_valid_zip(path) -> bool:
    try:
        with _zipfile.ZipFile(path):
            return True
    except _zipfile.BadZipFile:
        return False

def download_file(inf_id, inf_seq, file_seq, output_path, referer, label="", is_zip=None):
    """서울 공공데이터 단일 파일 다운로드. 이미 존재하고 유효하면 건너뜀."""
    out = Path(output_path)
    check_zip = is_zip if is_zip is not None else out.suffix.lower() == ".zip"

    if out.exists() and out.stat().st_size > 100_000:
        if check_zip and not _is_valid_zip(out):
            print(f"  re-download (corrupt ZIP): {out.name}")
            out.unlink()
        else:
            print(f"  skip (exists): {out.name}")
            return True

    print(f"  downloading: {label or out.name} ...", end=" ", flush=True)
    result = subprocess.run(
        [
            "curl", "-L", "-s",
            "-e", referer,
            "-X", "POST",
            "-d", f"infId={inf_id}",
            "-d", f"infSeq={inf_seq}",
            "-d", f"seq={file_seq}",
            "-d", f"seqNo={file_seq}",
            DL_URL,
            "-o", str(out),
        ],
        capture_output=True,
    )
    if result.returncode != 0 or not out.exists() or out.stat().st_size < 100_000:
        print("FAIL (download error)")
        out.unlink(missing_ok=True)
        return False
    if check_zip and not _is_valid_zip(out):
        print("FAIL (server returned error page, not a valid ZIP)")
        out.unlink(missing_ok=True)
        return False
    size_mb = out.stat().st_size / 1_048_576
    print(f"OK ({size_mb:.1f} MB)")
    return True

print("다운로드 함수 준비 완료")

In [6]:
# ── 4-1. 보조 데이터 4종 ──────────────────────────────────────────────
print("=== 보조 데이터 다운로드 ===")

aux_files = [
    ("OA-12914", "3", "153", DRIVE_RAW / "CARD_SUBWAY_MONTH_202604.csv",
     "https://data.seoul.go.kr/dataList/OA-12914/S/1/datasetView.do", "지하철 2026-04"),
    ("OA-12913", "3", "109", DRIVE_RAW / "bus_time_station_202604.csv",
     "https://data.seoul.go.kr/dataList/OA-12913/S/1/datasetView.do", "버스 2026-04"),
    ("OA-22160", "3", "1",   DRIVE_RAW / "seoul_admin_dong_area.zip",
     "https://data.seoul.go.kr/dataList/OA-22160/S/1/datasetView.do", "행정동 매핑"),
    ("OA-22266", "1", "30",  DRIVE_RAW / "seoul_living_interest_groups_202512.xlsx",
     "https://data.seoul.go.kr/dataList/OA-22266/F/1/datasetView.do", "시민생활 2025-12"),
]

for inf_id, inf_seq, file_seq, out_path, ref, label in aux_files:
    download_file(inf_id, inf_seq, file_seq, out_path, ref, label)
    time.sleep(1)

print("\n보조 데이터 완료")

=== 보조 데이터 다운로드 ===
  skip (exists): CARD_SUBWAY_MONTH_202604.csv
  skip (exists): bus_time_station_202604.csv
  skip (exists): seoul_admin_dong_area.zip
  skip (exists): seoul_living_interest_groups_202512.xlsx

보조 데이터 완료


In [7]:
from pathlib import Path
# ── 4-2. 생활이동 월말 스냅샷 39개 (2023-01 ~ 2026-03) ──────────────
import csv as _csv

MANIFEST_PATH = Path("/content/seoul_mobility/data_archive/metadata/living_migration_month_end_manifest.csv")

with MANIFEST_PATH.open() as f:
    reader = _csv.DictReader(f)
    month_end_entries = list(reader)

print(f"=== 월말 스냅샷 {len(month_end_entries)}개 다운로드 ===")
failed = []
for i, row in enumerate(month_end_entries, 1):
    yyyymm   = row["yyyymm"]
    filename = row["filename"]
    seq      = row["seq"]
    out_path = DRIVE_RAW / filename
    print(f"[{i:02d}/{len(month_end_entries)}] {yyyymm}", end="  ")
    ok = download_file("OA-22299", "1", seq, out_path, REFERER_LIVING, filename)
    if not ok:
        failed.append(filename)
    time.sleep(0.5)

print(f"\n월말 스냅샷 완료. 실패: {len(failed)}개")
if failed:
    print("실패 파일:", failed)

=== 월말 스냅샷 39개 다운로드 ===
[01/39] 202301    skip (exists): seoul_purpose_admdong4_in_20230131.zip
[02/39] 202302    skip (exists): seoul_purpose_admdong4_in_20230228.zip
[03/39] 202303    skip (exists): seoul_purpose_admdong4_in_20230331.zip
[04/39] 202304    skip (exists): seoul_purpose_admdong4_in_20230430.zip
[05/39] 202305    skip (exists): seoul_purpose_admdong4_in_20230531.zip
[06/39] 202306    skip (exists): seoul_purpose_admdong4_in_20230630.zip
[07/39] 202307    skip (exists): seoul_purpose_admdong4_in_20230731.zip
[08/39] 202308    skip (exists): seoul_purpose_admdong4_in_20230831.zip
[09/39] 202309    skip (exists): seoul_purpose_admdong4_in_20230930.zip
[10/39] 202310    skip (exists): seoul_purpose_admdong4_in_20231031.zip
[11/39] 202311    skip (exists): seoul_purpose_admdong4_in_20231130.zip
[12/39] 202312    skip (exists): seoul_purpose_admdong4_in_20231231.zip
[13/39] 202401    skip (exists): seoul_purpose_admdong4_in_20240131.zip
[14/39] 202402    skip (exists): seoul_p

In [8]:
# ── 4-3. 2026년 3월 일별 30개 ────────────────────────────────────────
import datetime

print("=== 2026년 3월 일별 다운로드 ===")
start = datetime.date(2026, 3, 1)
end   = datetime.date(2026, 3, 31)
skip  = {datetime.date(2026, 3, 28)}  # 원천 목록에 없음

daily_dates = [
    start + datetime.timedelta(days=d)
    for d in range((end - start).days + 1)
    if (start + datetime.timedelta(days=d)) not in skip
]

failed_daily = []
for i, dt in enumerate(daily_dates, 1):
    seq      = dt.strftime("%y%m%d")           # e.g. 260301
    filename = f"seoul_purpose_admdong4_in_{dt.strftime('%Y%m%d')}.zip"
    out_path = DRIVE_RAW / filename
    print(f"[{i:02d}/{len(daily_dates)}] {dt}", end="  ")
    ok = download_file("OA-22299", "1", seq, out_path, REFERER_LIVING, filename)
    if not ok:
        failed_daily.append(filename)
    time.sleep(0.5)

print(f"\n3월 일별 완료. 실패: {len(failed_daily)}개")
if failed_daily:
    print("실패 파일:", failed_daily)

=== 2026년 3월 일별 다운로드 ===
[01/30] 2026-03-01    downloading: seoul_purpose_admdong4_in_20260301.zip ... OK (43.4 MB)
[02/30] 2026-03-02    downloading: seoul_purpose_admdong4_in_20260302.zip ... OK (36.0 MB)
[03/30] 2026-03-03    downloading: seoul_purpose_admdong4_in_20260303.zip ... OK (50.0 MB)
[04/30] 2026-03-04    downloading: seoul_purpose_admdong4_in_20260304.zip ... OK (50.8 MB)
[05/30] 2026-03-05    downloading: seoul_purpose_admdong4_in_20260305.zip ... OK (50.7 MB)
[06/30] 2026-03-06    downloading: seoul_purpose_admdong4_in_20260306.zip ... OK (51.2 MB)
[07/30] 2026-03-07    downloading: seoul_purpose_admdong4_in_20260307.zip ... OK (46.2 MB)
[08/30] 2026-03-08    downloading: seoul_purpose_admdong4_in_20260308.zip ... OK (39.3 MB)
[09/30] 2026-03-09    downloading: seoul_purpose_admdong4_in_20260309.zip ... OK (49.8 MB)
[10/30] 2026-03-10    downloading: seoul_purpose_admdong4_in_20260310.zip ... OK (50.9 MB)
[11/30] 2026-03-11    downloading: seoul_purpose_admdong4_in_2026

In [9]:
# 다운로드 현황 요약
files = list(DRIVE_RAW.iterdir())
total_mb = sum(f.stat().st_size for f in files) / 1_048_576
print(f"Drive raw/ 파일 수: {len(files)}개")
print(f"총 용량: {total_mb:.0f} MB ({total_mb/1024:.1f} GB)")

Drive raw/ 파일 수: 72개
총 용량: 3122 MB (3.0 GB)


In [ ]:
# ── 4-4. 기존 Drive ZIP 파일 유효성 검사 (corrupt 파일 삭제) ─────────
# Drive에 이미 있던 파일 중 HTML 오류 페이지로 저장된 것을 제거합니다.
print("=== Drive raw/ ZIP 유효성 검사 ===")
corrupt = []
for f in sorted(DRIVE_RAW.glob("*.zip")):
    if not _is_valid_zip(f):
        print(f"  CORRUPT → 삭제: {f.name}")
        f.unlink()
        corrupt.append(f.name)

if corrupt:
    print(f"\n{len(corrupt)}개 corrupt 파일 삭제됨. 위 파일들을 재다운로드하려면")
    print("다운로드 셀들(4-1 ~ 4-3)을 다시 실행하세요.")
else:
    print("모든 ZIP 파일이 유효합니다.")

## Step 5. 분석 실행

In [10]:
import os
os.environ["SEOUL_RAW_DIR"] = str(DRIVE_RAW)

!python3 /content/seoul_mobility/seoul_mobility_analysis.py

0. Reading administrative-dong mapping...
   saved: /content/seoul_mobility/output/processed/admin_dong_mapping.csv (425 rows)
0-1. Reading 2030 single-household residential data...
   saved: /content/seoul_mobility/output/processed/young_single_household_residential_summary.csv (424 rows)
1. Cleaning subway data...
   saved: /content/seoul_mobility/output/processed/subway_station_daily.csv (18,522 rows)
2. Cleaning bus data...
   saved: /content/seoul_mobility/output/processed/bus_stop_route_summary.csv (42,096 rows)
   saved: /content/seoul_mobility/output/processed/bus_stop_route_hourly.csv (968,208 rows)
3. Analyzing 2030 living-migration OD data...
   reading living migration: seoul_purpose_admdong4_in_20260301.zip
   reading living migration: seoul_purpose_admdong4_in_20260302.zip
   reading living migration: seoul_purpose_admdong4_in_20260303.zip
   reading living migration: seoul_purpose_admdong4_in_20260304.zip
   reading living migration: seoul_purpose_admdong4_in_20260305.zi

## Step 6. 결과를 Drive에 저장

In [11]:
import shutil

shutil.copytree(
    "/content/seoul_mobility/output",
    str(DRIVE_OUTPUT),
    dirs_exist_ok=True,
)
print("결과 저장 완료:", DRIVE_OUTPUT)

결과 저장 완료: /content/drive/MyDrive/seoul_mobility/output


## Step 7. 시각화

월별 추세, 히트맵, 순위 변화, candidate_type 분포 등을 차례로 그립니다.

In [12]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager

# ── 한글 폰트 설치 ──────────────────────────────────────────────────
!apt-get install -y -q fonts-nanum 2>/dev/null
font_manager.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

OUTPUT = Path("/content/seoul_mobility/output")
VIZ_DIR = OUTPUT / "reports" / "viz"
VIZ_DIR.mkdir(parents=True, exist_ok=True)

# 데이터 로드
monthly   = pd.read_csv(OUTPUT / "processed/monthly_living_migration_2030_summary.csv", dtype={"d_admdong_cd": str})
trend     = pd.read_csv(OUTPUT / "processed/monthly_candidate_trend_summary.csv", dtype={"d_admdong_cd": str})
visitor   = pd.read_csv(OUTPUT / "processed/visitor_candidate_summary.csv", dtype={"d_admdong_cd": str})
mixed     = pd.read_csv(OUTPUT / "processed/mixed_commercial_residential_summary.csv", dtype={"d_admdong_cd": str})

monthly["yyyymm"] = monthly["yyyymm"].astype(str)
months_sorted = sorted(monthly["yyyymm"].unique())

print(f"월별 데이터: {monthly['yyyymm'].nunique()}개월 / {monthly['d_admdong_cd'].nunique()}개 행정동")
print(f"기간: {months_sorted[0]} ~ {months_sorted[-1]}")

Reading package lists...
Building dependency tree...
Reading state information...
The following NEW packages will be installed:
  fonts-nanum
0 upgraded, 1 newly installed, 0 to remove and 100 not upgraded.
Need to get 10.3 MB of archives.
After this operation, 34.1 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 fonts-nanum all 20200506-1 [10.3 MB]
Fetched 10.3 MB in 0s (37.5 MB/s)
Selecting previously unselected package fonts-nanum.
(Reading database ... 122402 files and directories currently installed.)
Preparing to unpack .../fonts-nanum_20200506-1_all.deb ...
Unpacking fonts-nanum (20200506-1) ...
Setting up fonts-nanum (20200506-1) ...
Processing triggers for fontconfig (2.13.1-4.2ubuntu5) ...


FileNotFoundError: [Errno 2] No such file or directory: '/content/seoul_mobility/output/processed/monthly_living_migration_2030_summary.csv'

In [ ]:
# ── 차트 1: 방문성 후보 Top 15 월별 adjusted_mobility_score 추세 ────
TOP_N = 15

# 최신 월 기준 상위 행정동 선정 (방문성 검토 + 혼재형)
latest_month = months_sorted[-1]
top_dongs = (
    monthly[
        (monthly["yyyymm"] == latest_month)
        & monthly["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
    ]
    .sort_values("adjusted_mobility_score", ascending=False)
    .head(TOP_N)["d_admdong_cd"]
    .tolist()
)

fig, ax = plt.subplots(figsize=(16, 6))
cmap = plt.cm.get_cmap("tab20", TOP_N)

for i, cd in enumerate(top_dongs):
    sub = monthly[monthly["d_admdong_cd"] == cd].sort_values("yyyymm")
    name = sub["d_admdong_name"].iloc[0] if "d_admdong_name" in sub.columns else cd
    ax.plot(sub["yyyymm"], sub["adjusted_mobility_score"], marker="o", markersize=3,
            linewidth=1.5, label=name, color=cmap(i))

ax.set_title(f"방문성 후보 Top {TOP_N} 월별 adjusted_mobility_score 추세", fontsize=14, pad=12)
ax.set_xlabel("월")
ax.set_ylabel("adjusted_mobility_score")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
tick_every = max(1, len(months_sorted) // 12)
ax.set_xticks(months_sorted[::tick_every])
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(VIZ_DIR / "01_top15_monthly_score_trend.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 차트 2: 행정동 × 월 히트맵 (방문성 후보 Top 30) ─────────────────
TOP_HEAT = 30

top_heat_dongs = (
    trend[
        trend["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
    ]
    .sort_values("avg_adjusted_mobility_score", ascending=False)
    .head(TOP_HEAT)
)

dong_order = top_heat_dongs["d_admdong_cd"].tolist()
name_map   = top_heat_dongs.set_index("d_admdong_cd")["d_admdong_name"].to_dict() \
             if "d_admdong_name" in top_heat_dongs.columns else {cd: cd for cd in dong_order}

heat_df = (
    monthly[monthly["d_admdong_cd"].isin(dong_order)]
    .pivot_table(index="d_admdong_cd", columns="yyyymm", values="adjusted_mobility_score")
    .reindex(dong_order)
)
heat_df.index = [name_map.get(cd, cd) for cd in heat_df.index]

fig, ax = plt.subplots(figsize=(max(16, len(months_sorted) * 0.45), 10))
im = ax.imshow(heat_df.values, aspect="auto", cmap="RdYlGn", interpolation="nearest")
plt.colorbar(im, ax=ax, shrink=0.8, label="adjusted_mobility_score")

ax.set_yticks(range(len(heat_df.index)))
ax.set_yticklabels(heat_df.index, fontsize=9)
ax.set_xticks(range(len(heat_df.columns)))
ax.set_xticklabels(heat_df.columns, rotation=90, fontsize=8)
ax.set_title(f"방문성 후보 Top {TOP_HEAT} 행정동 × 월별 Score 히트맵", fontsize=14, pad=12)
plt.tight_layout()
fig.savefig(VIZ_DIR / "02_heatmap_dong_month.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 차트 3: Score Slope 상위/하위 15개 (상승/하락 추세) ────────────
trend_visitor = trend[
    trend["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
].copy()

rising  = trend_visitor.nlargest(15, "score_slope")
falling = trend_visitor.nsmallest(15, "score_slope")
combined = pd.concat([rising, falling]).drop_duplicates().sort_values("score_slope")

if "d_admdong_name" in combined.columns:
    labels = combined["d_admdong_name"].tolist()
else:
    labels = combined["d_admdong_cd"].tolist()
slopes = combined["score_slope"].tolist()
colors = ["#d62728" if s < 0 else "#2ca02c" for s in slopes]

fig, ax = plt.subplots(figsize=(10, 10))
bars = ax.barh(range(len(labels)), slopes, color=colors)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Score Slope (월 단위 기울기)")
ax.set_title("방문성 후보 행정동 Score 추세 — 상승 Top15 / 하락 Top15", fontsize=13, pad=10)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(VIZ_DIR / "03_score_slope_ranking.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 차트 4: 최신 월 방문성 후보 Top 20 (horizontal bar) ───────────
latest_visitor = (
    monthly[
        (monthly["yyyymm"] == latest_month)
        & monthly["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
    ]
    .sort_values("adjusted_mobility_score", ascending=False)
    .head(20)
)

name_col = "d_admdong_name" if "d_admdong_name" in latest_visitor.columns else "d_admdong_cd"
filter_colors = {
    "방문성 검토": "#1f77b4",
    "혼재형 (상권+거주)": "#ff7f0e",
}
bar_colors = [filter_colors.get(r, "#7f7f7f") for r in latest_visitor["residential_filter"]]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(
    range(len(latest_visitor)),
    latest_visitor["adjusted_mobility_score"].values[::-1] if False else latest_visitor["adjusted_mobility_score"].values,
    color=bar_colors,
)
ax.set_yticks(range(len(latest_visitor)))
ax.set_yticklabels(latest_visitor[name_col].tolist(), fontsize=9)
ax.invert_yaxis()
ax.set_xlabel("adjusted_mobility_score")
ax.set_title(f"최신 월({latest_month}) 방문성 후보 Top 20", fontsize=13, pad=10)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=v, label=k) for k, v in filter_colors.items()]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(VIZ_DIR / "04_latest_month_top20.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 차트 5: candidate_type × residential_filter 분포 (stacked bar) ──
if "candidate_type" in monthly.columns:
    latest_all = monthly[monthly["yyyymm"] == latest_month].copy()
    ct_rf = (
        latest_all.groupby(["candidate_type", "residential_filter"])
        .size()
        .unstack(fill_value=0)
    )
    ct_rf_pct = ct_rf.div(ct_rf.sum(axis=1), axis=0) * 100

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # 절대 수
    ct_rf.plot(kind="bar", stacked=True, ax=axes[0], colormap="Set2")
    axes[0].set_title("candidate_type × residential_filter 분포 (개수)", fontsize=12)
    axes[0].set_xlabel("")
    axes[0].tick_params(axis="x", rotation=30)
    axes[0].legend(fontsize=8)

    # 비율
    ct_rf_pct.plot(kind="bar", stacked=True, ax=axes[1], colormap="Set2")
    axes[1].set_title("candidate_type × residential_filter 분포 (%)", fontsize=12)
    axes[1].set_xlabel("")
    axes[1].set_ylabel("비율 (%)")
    axes[1].tick_params(axis="x", rotation=30)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    fig.savefig(VIZ_DIR / "05_candidate_type_distribution.png", bbox_inches="tight")
    plt.show()
else:
    print("candidate_type 컬럼 없음 — 이 차트는 건너뜁니다")

In [ ]:
# ── 차트 6: 월별 전체 2030 유입량 추이 (서울 전체 합계) ─────────────
monthly_total = (
    monthly.groupby("yyyymm")["cnt_2030"]
    .sum()
    .reset_index()
    .rename(columns={"cnt_2030": "total_2030"})
)

# 주말 스냅샷 강조
weekend_flag = "is_weekend_snapshot" in monthly.columns
if weekend_flag:
    weekend_months = set(
        monthly[monthly["is_weekend_snapshot"] == True]["yyyymm"].unique()
    )

fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(monthly_total["yyyymm"], monthly_total["total_2030"],
        marker="o", markersize=4, linewidth=1.5, color="#1f77b4", label="전체 2030 유입 합계")

if weekend_flag:
    wknd = monthly_total[monthly_total["yyyymm"].isin(weekend_months)]
    ax.scatter(wknd["yyyymm"], wknd["total_2030"],
               color="orange", zorder=5, s=60, label="주말 스냅샷")

ax.set_title("서울 전체 월별 2030 유입량 추이 (월말 스냅샷 기준)", fontsize=13, pad=10)
ax.set_xlabel("월")
ax.set_ylabel("2030 유입 인원 (합계)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
tick_every = max(1, len(months_sorted) // 12)
ax.set_xticks(months_sorted[::tick_every])
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(VIZ_DIR / "06_total_2030_monthly_trend.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── 차트 7: 상위 10개 행정동 월별 순위 변화 (Bump Chart) ────────────
TOP_BUMP = 10

# 방문성 후보만, 월별 순위 계산
visitor_monthly = monthly[
    monthly["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
].copy()
visitor_monthly["visitor_rank"] = visitor_monthly.groupby("yyyymm")["adjusted_mobility_score"].rank(
    ascending=False, method="min"
)

# 최신 월 기준 상위 행정동
top_bump_dongs = (
    visitor_monthly[visitor_monthly["yyyymm"] == latest_month]
    .sort_values("visitor_rank")
    .head(TOP_BUMP)["d_admdong_cd"]
    .tolist()
)

name_col = "d_admdong_name" if "d_admdong_name" in visitor_monthly.columns else "d_admdong_cd"

fig, ax = plt.subplots(figsize=(16, 6))
cmap = plt.cm.get_cmap("tab10", TOP_BUMP)

for i, cd in enumerate(top_bump_dongs):
    sub = (
        visitor_monthly[visitor_monthly["d_admdong_cd"] == cd]
        .sort_values("yyyymm")
    )
    name = sub[name_col].iloc[0]
    ax.plot(sub["yyyymm"], sub["visitor_rank"], marker="o", markersize=4,
            linewidth=1.5, label=name, color=cmap(i))

ax.invert_yaxis()
ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.set_title(f"방문성 후보 Top {TOP_BUMP} 월별 순위 변화 (1위 = 상단)", fontsize=13, pad=10)
ax.set_xlabel("월")
ax.set_ylabel("방문성 후보 순위")
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1), fontsize=8)
tick_every = max(1, len(months_sorted) // 12)
ax.set_xticks(months_sorted[::tick_every])
ax.tick_params(axis="x", rotation=45)
ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(VIZ_DIR / "07_bump_chart_visitor_rank.png", bbox_inches="tight")
plt.show()

In [ ]:
# 시각화 파일을 Drive에도 저장
import shutil
drive_viz = DRIVE_OUTPUT / "reports" / "viz"
drive_viz.mkdir(parents=True, exist_ok=True)
for png in VIZ_DIR.glob("*.png"):
    shutil.copy(png, drive_viz / png.name)
print(f"차트 {len(list(VIZ_DIR.glob('*.png')))}개 Drive에 저장 완료:", drive_viz)

## 결과 요약

In [ ]:
print("===== 분석 결과 요약 =====")
print(f"분석 기간: {months_sorted[0]} ~ {months_sorted[-1]} ({len(months_sorted)}개월)")
print(f"분석 행정동 수: {monthly['d_admdong_cd'].nunique()}개")

latest_v = monthly[
    (monthly["yyyymm"] == latest_month)
    & (monthly["residential_filter"] == "방문성 검토")
]
latest_m = monthly[
    (monthly["yyyymm"] == latest_month)
    & (monthly["residential_filter"] == "혼재형 (상권+거주)")
]
latest_r = monthly[
    (monthly["yyyymm"] == latest_month)
    & (monthly["residential_filter"] == "2030 자취/거주성 높음")
]

print(f"\n최신 월({latest_month}) 분류 현황:")
print(f"  방문성 검토:        {len(latest_v)}개 행정동")
print(f"  혼재형 (상권+거주): {len(latest_m)}개 행정동")
print(f"  2030 거주성 높음:   {len(latest_r)}개 행정동")

print(f"\n방문성 후보 Top 5 ({latest_month}):")
name_col = "d_admdong_name" if "d_admdong_name" in monthly.columns else "d_admdong_cd"
top5 = (
    monthly[
        (monthly["yyyymm"] == latest_month)
        & monthly["residential_filter"].isin(["방문성 검토", "혼재형 (상권+거주)"])
    ]
    .sort_values("adjusted_mobility_score", ascending=False)
    .head(5)[[name_col, "residential_filter", "adjusted_mobility_score", "candidate_type"]]
)
display(top5)